# Dunnhumby: M1 조건부 CLV 후보 구별력 진단

기존 seed-42 M1 체크포인트를 그대로 사용합니다. 각 신규상품 정답마다 **같은 상품 인기도 10분위**에서 **M1 점수가 가장 가까운 미구매·비정답 상품 5개**를 대조상품으로 고른 뒤, N 적합도와 V 적합도가 정답을 더 높게 평가하는지 확인합니다.

새 학습·체크포인트 선택·재정렬·final test·holdout은 수행하지 않습니다. 결과는 다음 M2·M3·M4 설계의 기제 진단이며 추천성능 결과가 아닙니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REVIEWED_SHA = 'e8abfcb'
%cd /content
!rm -rf /content/clv-m2-lightgcn-runner
!git clone -q https://github.com/jung-un/clv-m2-lightgcn-runner.git
%cd /content/clv-m2-lightgcn-runner
!git checkout -q $REVIEWED_SHA
import subprocess
assert subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip().startswith(REVIEWED_SHA)


In [ ]:
import importlib
import json
import torch
import lightgcn_clv_incremental_candidate_signal_diagnostic as diagnostic
diagnostic = importlib.reload(diagnostic)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
assert diagnostic.CODE_VERSION == 'clv-incremental-candidate-signal-diagnostic-v1'
cfg = diagnostic.configure_incremental_candidate_signal_diagnostic('dunnhumby')
print(json.dumps(diagnostic.preflight_summary(cfg), ensure_ascii=False, indent=2))


In [ ]:
diagnostic = importlib.reload(diagnostic)
cfg = diagnostic.configure_incremental_candidate_signal_diagnostic('dunnhumby')
paths = diagnostic.run_incremental_candidate_signal_diagnostic(cfg)


In [ ]:
import json
import pandas as pd
from IPython.display import display

summary = pd.read_csv(paths['summary_csv'])
report = json.load(open(paths['json']))
print('1) 전체·CLV 구간별 조건부 후보 구별력')
display(summary)
print('2) 사용자 단위 bootstrap 95% 구간')
display(pd.DataFrame(report['bootstrap']['signals']).T)
print('3) 대조상품 매칭 품질')
print(json.dumps(report['matching_diagnostics'], ensure_ascii=False, indent=2))
print('4) q_C 자체의 후보 구별력')
print(json.dumps(report['q_c_candidate_diagnostic'], ensure_ascii=False, indent=2))
print('판독: 한 신호의 사용자 평균 균형승률 95% 구간이 0.5보다 높고 H&M에서도 같은 방향일 때만 후보별 추가정보로 기록합니다.')
print('이 조건을 통과해도 해당 신호를 넣은 M2·M3·M4의 성능개선이 입증되는 것은 아닙니다.')
print('결과 파일:', paths)
